# Walkthrough: paired-timepoint scRNA-seq of the zebrafish eye

This notebook calls the same functions the CLI does. The pipeline itself lives in
`src/screye/` so that the analysis is testable and re-runnable; this notebook is
for inspecting intermediate state and trying alternatives interactively.

Run `python tests/fixtures/make_synthetic_h5.py` first if you do not have the real data.

In [ ]:
from pathlib import Path

import logging
import os
import pandas as pd
import scanpy as sc

from screye.cli import find_repo_root
from screye.config import Config
from screye.cluster import load_markers, process
from screye.qc import build_qc_dataset, load_sample

logging.basicConfig(level=logging.INFO, format="%(levelname)-7s | %(message)s")
sc.set_figure_params(dpi=100, facecolor="white")

# `find_repo_root` walks up for pyproject.toml, so this cell works whether the
# notebook is launched from notebooks/, from the repo root, or under papermill.
# It is the same function the CLI uses - one definition, one behaviour.
REPO = find_repo_root()

# SCREYE_CONFIG lets the test suite (and anyone else) point this walkthrough at
# a different config - the notebook test uses it to run on synthetic data, so
# executing the notebook needs neither the real libraries nor their runtime.
CONFIG = Path(os.environ.get("SCREYE_CONFIG") or REPO / "config" / "config.yaml")

cfg = Config.from_yaml(CONFIG).resolve(REPO)
cfg.validate_inputs()

# Load both libraries straight from data/. build_qc_dataset() re-reads them in
# the next cell, deliberately: the notebook then takes exactly the same code
# path as `pixi run run`, so nothing can be true here and false in the pipeline.
raw = {spec.name: load_sample(spec) for spec in cfg.samples}

pd.DataFrame(
    [
        {
            "sample": spec.name,
            "timepoint": spec.timepoint,
            "cells": raw[spec.name].n_obs,
            "genes": raw[spec.name].n_vars,
            "file": spec.h5.name,
            "MB": round(spec.h5.stat().st_size / 1e6, 1),
        }
        for spec in cfg.samples
    ]
).set_index("sample")

## 1. Load and QC

Cells are flagged, plotted, then removed — in that order, so it is visible when
a whole population is about to be dropped.

In [ ]:
adata = build_qc_dataset(cfg)
adata

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
             groupby="sample", rotation=45, multi_panel=True)

## 2. Normalise, embed, cluster, annotate

`process` returns the analysed object, the cluster x cell-type score matrix
behind the labels, and the full Wilcoxon differential expression table.

In [ ]:
adata, scores, de = process(adata, cfg)
scores.round(3)

The score matrix is the audit trail for the annotation. A row whose top two
values are close is a cluster the marker panel cannot resolve — those are
labelled `(low confidence)` rather than given a clean name.

In [ ]:
sc.pl.umap(adata, color=["leiden", "cell_type", "sample"], wspace=0.35, ncols=2)

In [ ]:
sc.pl.tsne(adata, color=["leiden", "cell_type", "sample"], wspace=0.35, ncols=2)

## 3. Marker evidence

The curated panel and the data-driven top genes should agree. Where they do not,
the data-driven genes are the ones to follow up.

In [ ]:
markers = load_markers(cfg.markers_file)
present = {k: [g for g in v if g in adata.raw.var_names] for k, v in markers.items()}
sc.pl.dotplot(adata, {k: v for k, v in present.items() if v},
              groupby="leiden", use_raw=True, standard_scale="var")

In [ ]:
de.query("group == '0'").nlargest(10, "scores")[["names", "scores", "logfoldchanges", "pvals_adj"]]

## 4. Composition across timepoints

Descriptive only: one library per timepoint means there is no biological
replication, so a proportional shift here cannot be tested statistically.

In [ ]:
import pandas as pd
pd.crosstab(adata.obs["cell_type"], adata.obs["sample"])